# Module 05: Simple Linear Regression
*Statistics for Econometrics — An intermediate course for researchers*

This module covers the foundations of linear regression: OLS estimation, goodness of fit, inference on coefficients, residual diagnostics, and the Gauss-Markov assumptions. We use examples from regional economics and logistics infrastructure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
except ImportError:
    !pip install statsmodels
    import statsmodels.api as sm
    from statsmodels.formula.api import ols

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 5.1 From Groups to Gradients

In previous modules, we tested whether group means differ (ANOVA, t-tests). Linear regression extends this logic: instead of comparing discrete groups, we model how a continuous outcome **Y** changes in relation to a continuous predictor **X**.

The simple linear regression model is:

$$Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i$$

where:
- **β₀** (intercept): the expected value of Y when X = 0
- **β₁** (slope): the change in Y for a one-unit increase in X
- **εᵢ** (error): the deviation of observation i from the line

### Association vs Causation

A regression coefficient tells us about **association**—how X and Y co-vary in the data. It does not automatically imply causation. Causation requires (1) temporal precedence, (2) no confounders, and (3) a plausible mechanism. Always think about the source of your data and whether the regression design can support causal claims.

In [ ]:
# Generate dataset: 30 municipalities, distance from logistics hub and commercial property value
# Commercial property closer to the hub commands higher value (better logistics access)
np.random.seed(42)
n = 30
distance = np.random.uniform(2, 50, n)

# Property value decreases with distance: closer to hub = higher value
# Intercept ≈ 350,000; slope ≈ -1,200 per km
property_value = 350000 - 1200 * distance + np.random.normal(0, 25000, n)

df = pd.DataFrame({
    'distance_km': distance,
    'property_value': property_value
})

print("First 5 observations:")
print(df.head())
print(f"\nDataset shape: {df.shape}")
print(f"Distance range: {df['distance_km'].min():.1f} to {df['distance_km'].max():.1f} km")
print(f"Property value range: £{df['property_value'].min():.0f} to £{df['property_value'].max():.0f}")

In [ ]:
# Visualise the raw data
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['distance_km'], df['property_value']/1000, s=100, alpha=0.6, color='steelblue', edgecolors='navy')
ax.set_xlabel('Distance from Logistics Hub (km)', fontsize=12)
ax.set_ylabel('Commercial Property Value (£000s)', fontsize=12)
ax.set_title('Commercial Property Value vs Distance from Logistics Hub\n30 Municipalities', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice the downward trend: as distance increases, property value tends to decrease.")

## 5.2 OLS Estimation

Ordinary Least Squares (OLS) is the most widely used method for fitting a regression line. The goal is to choose β₀ and β₁ to minimise the sum of squared residuals:

$$\min_{\beta_0, \beta_1} \sum_{i=1}^{n} (Y_i - \beta_0 - \beta_1 X_i)^2$$

The OLS formulae are:

$$\hat{\beta}_1 = \frac{\text{Cov}(X, Y)}{\text{Var}(X)} = \frac{\sum (X_i - \bar{X})(Y_i - \bar{Y})}{\sum (X_i - \bar{X})^2}$$

$$\hat{\beta}_0 = \bar{Y} - \hat{\beta}_1 \bar{X}$$

Notice that the fitted line always passes through the point of means $(\bar{X}, \bar{Y})$.

In [ ]:
# Manual OLS calculation step by step
X = df['distance_km'].values
Y = df['property_value'].values

# Calculate means
X_mean = np.mean(X)
Y_mean = np.mean(Y)
print(f"Mean distance (X̄): {X_mean:.2f} km")
print(f"Mean property value (Ȳ): £{Y_mean:.0f}")

# Calculate deviations
X_dev = X - X_mean
Y_dev = Y - Y_mean

# Calculate covariance and variance
cov_XY = np.sum(X_dev * Y_dev) / n
var_X = np.sum(X_dev**2) / n
print(f"\nCovariance Cov(X,Y): {cov_XY:.2f}")
print(f"Variance Var(X): {var_X:.2f}")

# OLS estimates
beta_1_manual = cov_XY / var_X
beta_0_manual = Y_mean - beta_1_manual * X_mean

print(f"\n--- OLS Estimates (Manual Calculation) ---")
print(f"β̂₁ (slope):      {beta_1_manual:.2f} (£ per km)")
print(f"β̂₀ (intercept):  £{beta_0_manual:.0f}")
print(f"\nInterpretation: For every additional kilometre from the logistics hub,")
print(f"commercial property value decreases by £{abs(beta_1_manual):.0f}.")

In [ ]:
# Verify with scipy.stats.linregress
slope_scipy, intercept_scipy, r_value_scipy, p_value_scipy, se_scipy = stats.linregress(X, Y)
print("Verification with scipy.stats.linregress:")
print(f"Slope: {slope_scipy:.2f}")
print(f"Intercept: £{intercept_scipy:.0f}")
print(f"R-value: {r_value_scipy:.4f}")
print(f"\nOur manual calculation matches scipy!")

In [ ]:
# Verify with statsmodels OLS
X_with_const = sm.add_constant(X)
model = sm.OLS(Y, X_with_const)
results = model.fit()
print(results.summary())

print(f"\nCoefficients from statsmodels:")
print(f"β̂₀: £{results.params[0]:.0f}")
print(f"β̂₁: £{results.params[1]:.2f} per km")

In [ ]:
# Fitted values and visualisation
Y_fitted = beta_0_manual + beta_1_manual * X

fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot
ax.scatter(X, Y/1000, s=100, alpha=0.6, color='steelblue', edgecolors='navy', label='Observed data')

# Fitted line
ax.plot(X, Y_fitted/1000, 'r-', linewidth=2.5, label='Fitted regression line')

# Mark point of means
ax.plot(X_mean, Y_mean/1000, 'g*', markersize=20, label=f'Point of means (X̄, Ȳ)', zorder=5)

# Annotations
ax.text(X_mean + 2, Y_mean/1000 + 5, f'(X̄={X_mean:.1f}, Ȳ=£{Y_mean/1000:.0f}k)', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
ax.text(5, 380, f'β̂₀ = £{beta_0_manual:.0f}\nβ̂₁ = £{beta_1_manual:.0f}/km', fontsize=11, bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.8))

ax.set_xlabel('Distance from Logistics Hub (km)', fontsize=12)
ax.set_ylabel('Commercial Property Value (£000s)', fontsize=12)
ax.set_title('OLS Regression Line with Point of Means', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.3 Goodness of Fit: R²

How well does the regression line fit the data? We decompose total variation in Y:

$$\text{SS}_{\text{tot}} = \sum (Y_i - \bar{Y})^2 = \text{SS}_{\text{reg}} + \text{SS}_{\text{res}}$$

where:
- **SS_tot** = total sum of squares (variation around the mean)
- **SS_reg** = regression sum of squares (variation explained by X)
- **SS_res** = residual sum of squares (variation not explained)

The coefficient of determination is:

$$R^2 = \frac{\text{SS}_{\text{reg}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}$$

**R²** ranges from 0 to 1 and represents the proportion of variation in Y explained by X. A low R² does not mean the model is useless—it might still provide valid inference on β₁—but it suggests that X alone cannot predict Y precisely.

In [ ]:
# Calculate R² manually
residuals = Y - Y_fitted

SS_tot = np.sum((Y - Y_mean)**2)
SS_reg = np.sum((Y_fitted - Y_mean)**2)
SS_res = np.sum(residuals**2)

print(f"Sum of Squares Decomposition:")
print(f"SS_tot (total):      {SS_tot:.2e}")
print(f"SS_reg (explained):  {SS_reg:.2e}")
print(f"SS_res (residual):   {SS_res:.2e}")
print(f"\nVerification: SS_reg + SS_res = {SS_reg + SS_res:.2e}")
print(f"(should equal SS_tot = {SS_tot:.2e})")

# Calculate R² both ways
R2_method1 = SS_reg / SS_tot
R2_method2 = 1 - (SS_res / SS_tot)
R2_correlation = r_value_scipy**2  # R² = r²

print(f"\n--- R² Estimates ---")
print(f"Method 1 (SS_reg/SS_tot):     {R2_method1:.4f}")
print(f"Method 2 (1 - SS_res/SS_tot): {R2_method2:.4f}")
print(f"From correlation (r²):        {R2_correlation:.4f}")
print(f"From statsmodels:             {results.rsquared:.4f}")

print(f"\nInterpretation: {R2_method1*100:.1f}% of variation in property value")
print(f"is explained by distance from the logistics hub.")

In [ ]:
# Visualise the SS decomposition for one observation
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot and regression line
ax.scatter(X, Y/1000, s=80, alpha=0.5, color='steelblue', edgecolors='navy')
ax.plot(X, Y_fitted/1000, 'r-', linewidth=2.5, label='Fitted line', zorder=3)
ax.axhline(y=Y_mean/1000, color='green', linestyle='--', linewidth=1.5, label='Mean of Y', alpha=0.7)

# Highlight one observation (e.g., index 5)
i = 5
ax.scatter(X[i], Y[i]/1000, s=200, color='red', edgecolors='darkred', zorder=5, marker='D')

# Draw deviation lines
# Total deviation: from Y_mean to Y_i
ax.plot([X[i], X[i]], [Y_mean/1000, Y[i]/1000], 'g-', linewidth=2.5, alpha=0.7, label='Total deviation (SS_tot)')

# Explained deviation: from Y_mean to fitted value
ax.plot([X[i], X[i]], [Y_mean/1000, Y_fitted[i]/1000], 'orange', linewidth=2.5, alpha=0.8, label='Explained deviation (SS_reg)')

# Residual: from fitted to actual
ax.plot([X[i], X[i]], [Y_fitted[i]/1000, Y[i]/1000], 'purple', linewidth=2.5, alpha=0.8, label='Residual (SS_res)')

ax.set_xlabel('Distance from Logistics Hub (km)', fontsize=12)
ax.set_ylabel('Commercial Property Value (£000s)', fontsize=12)
ax.set_title('SS Decomposition: Total = Explained + Residual', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"For observation {i}: (X={X[i]:.1f}, Y=£{Y[i]:.0f})")
print(f"  Total deviation from mean:     £{Y[i] - Y_mean:.0f}")
print(f"  Explained by regression:       £{Y_fitted[i] - Y_mean:.0f}")
print(f"  Residual (unexplained):        £{residuals[i]:.0f}")

## 5.4 Inference on β̂₁

We now want to test hypotheses and construct confidence intervals around our slope estimate.

Under the Gauss-Markov assumptions (covered in Section 5.6), the sampling distribution of β̂₁ is approximately normal:

$$\hat{\beta}_1 \sim N\left(\beta_1, \text{SE}(\hat{\beta}_1)^2\right)$$

The standard error is:

$$\text{SE}(\hat{\beta}_1) = \frac{s_e}{\sqrt{\sum (X_i - \bar{X})^2}}$$

where $s_e = \sqrt{\frac{\text{SS}_{\text{res}}}{n-2}}$ is the residual standard error.

We test the null hypothesis **H₀: β₁ = 0** (no relationship) using a t-statistic:

$$t = \frac{\hat{\beta}_1}{\text{SE}(\hat{\beta}_1)} \sim t_{n-2}$$

The 95% confidence interval is:

$$\text{CI} = \hat{\beta}_1 \pm t_{\alpha/2, n-2} \times \text{SE}(\hat{\beta}_1)$$

In [ ]:
# Extract inference statistics from statsmodels
print("=" * 70)
print("STATSMODELS REGRESSION SUMMARY")
print("=" * 70)
print(results.summary())
print("\n" + "=" * 70)

In [ ]:
# Manual calculation of standard error and t-statistic
s_e = np.sqrt(SS_res / (n - 2))  # Residual standard error
sum_X_dev_sq = np.sum((X - X_mean)**2)
SE_beta1 = s_e / np.sqrt(sum_X_dev_sq)

t_stat = beta_1_manual / SE_beta1
p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), n - 2))  # Two-tailed

print(f"--- Manual Calculation of Inference Statistics ---")
print(f"Residual standard error (s_e): £{s_e:.2f}")
print(f"Sum of squared deviations:      {sum_X_dev_sq:.2f}")
print(f"\nStandard Error of β̂₁:          £{SE_beta1:.2f}")
print(f"t-statistic:                    {t_stat:.4f}")
print(f"p-value (two-tailed):           {p_value:.6f}")

# Verify against statsmodels
print(f"\n--- Verification against statsmodels ---")
print(f"SE from statsmodels:            £{results.bse[1]:.2f}")
print(f"t-stat from statsmodels:        {results.tvalues[1]:.4f}")
print(f"p-value from statsmodels:       {results.pvalues[1]:.6f}")

In [ ]:
# Construct 95% confidence interval for β₁
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, n - 2)
ci_lower = beta_1_manual - t_crit * SE_beta1
ci_upper = beta_1_manual + t_crit * SE_beta1

print(f"95% Confidence Interval for β₁:")
print(f"[£{ci_lower:.2f}, £{ci_upper:.2f}]")
print(f"\nInterpretation: We are 95% confident that the true effect of")
print(f"distance on property value lies between £{ci_lower:.2f} and £{ci_upper:.2f} per km.")
print(f"\nSince zero is not in the CI, we reject H₀: β₁ = 0 at the 5% level.")
print(f"There is strong evidence that distance is associated with property value.")

# Verify against statsmodels
ci_from_results = results.conf_int(alpha=0.05)
print(f"\nFrom statsmodels:")
print(f"[£{ci_from_results[1, 0]:.2f}, £{ci_from_results[1, 1]:.2f}]")

In [ ]:
# Simulation: sampling distribution of β̂₁
# In practice, we only have one sample. But if we could repeatedly sample from
# the population, β̂₁ would vary. Under the Gauss-Markov assumptions, it's approximately normal.

np.random.seed(42)
n_simulations = 10000
beta1_estimates = []

for sim in range(n_simulations):
    # Resample from the original data (bootstrap approach)
    idx = np.random.choice(n, n, replace=True)
    X_boot = X[idx]
    Y_boot = Y[idx]
    
    # Estimate β₁
    X_mean_boot = np.mean(X_boot)
    Y_mean_boot = np.mean(Y_boot)
    cov_boot = np.mean((X_boot - X_mean_boot) * (Y_boot - Y_mean_boot))
    var_boot = np.mean((X_boot - X_mean_boot)**2)
    beta1_boot = cov_boot / var_boot
    beta1_estimates.append(beta1_boot)

beta1_estimates = np.array(beta1_estimates)

# Plot the sampling distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(beta1_estimates, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='navy')

# Overlay normal distribution
x_range = np.linspace(beta1_estimates.min(), beta1_estimates.max(), 100)
ax.plot(x_range, stats.norm.pdf(x_range, np.mean(beta1_estimates), np.std(beta1_estimates)), 
        'r-', linewidth=2, label='Normal distribution')

ax.axvline(beta_1_manual, color='green', linestyle='--', linewidth=2, label=f'Our estimate (£{beta_1_manual:.0f})')
ax.axvline(np.mean(beta1_estimates), color='orange', linestyle='--', linewidth=2, label=f'Mean of simulation (£{np.mean(beta1_estimates):.0f})')

ax.set_xlabel('Estimated β₁ (£ per km)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Sampling Distribution of β̂₁ (Bootstrap Simulation)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Sampling Distribution Summary:")
print(f"Mean of β̂₁ estimates:       £{np.mean(beta1_estimates):.2f}")
print(f"SD of β̂₁ estimates:         £{np.std(beta1_estimates):.2f}")
print(f"Our SE calculation:          £{SE_beta1:.2f}")
print(f"\nThe simulated SD closely matches our calculated SE,")
print(f"confirming the asymptotic normality of β̂₁.")

## 5.5 Residual Analysis

After fitting a regression, we examine the residuals $\hat{\varepsilon}_i = Y_i - \hat{Y}_i$ to diagnose potential model violations.

Key diagnostic plots:
1. **Residuals vs Fitted Values**: Check linearity and homoscedasticity (constant variance)
2. **Q-Q Plot**: Check normality of errors
3. **Residuals vs X**: Visualise patterns missed by the line
4. **Histogram of Residuals**: Assess distributional shape

**Red flags to watch for:**
- Non-random patterns (e.g., U-shaped scatter) → linearity violated
- Increasing spread → heteroscedasticity (non-constant variance)
- Deviations from the Q-Q line → non-normality
- Outliers or high-leverage points → sensitivity analysis needed

In [ ]:
# Create 2x2 diagnostic plot panel
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Fitted Values
ax = axes[0, 0]
ax.scatter(Y_fitted, residuals, s=80, alpha=0.6, color='steelblue', edgecolors='navy')
ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Fitted Values (£)', fontsize=10)
ax.set_ylabel('Residuals (£)', fontsize=10)
ax.set_title('1. Residuals vs Fitted Values', fontsize=11, fontweight='bold')
ax.text(0.05, 0.95, 'Look for:\n• Random scatter\n• No patterns', 
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
ax.grid(True, alpha=0.3)

# 2. Q-Q Plot
ax = axes[0, 1]
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title('2. Normal Q-Q Plot', fontsize=11, fontweight='bold')
ax.text(0.05, 0.95, 'Look for:\n• Points on the line\n• Linear trend', 
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
ax.grid(True, alpha=0.3)

# 3. Residuals vs X
ax = axes[1, 0]
ax.scatter(X, residuals, s=80, alpha=0.6, color='steelblue', edgecolors='navy')
ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Distance (km)', fontsize=10)
ax.set_ylabel('Residuals (£)', fontsize=10)
ax.set_title('3. Residuals vs X', fontsize=11, fontweight='bold')
ax.text(0.05, 0.95, 'Look for:\n• Uniform scatter\n• No trends', 
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
ax.grid(True, alpha=0.3)

# 4. Histogram of Residuals
ax = axes[1, 1]
ax.hist(residuals, bins=12, density=True, alpha=0.7, color='steelblue', edgecolor='navy')
# Overlay normal distribution
mu, sigma = residuals.mean(), residuals.std()
x = np.linspace(residuals.min(), residuals.max(), 100)
ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal fit')
ax.set_xlabel('Residuals (£)', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('4. Histogram of Residuals', fontsize=11, fontweight='bold')
ax.text(0.05, 0.95, 'Look for:\n• Bell shape\n• Symmetry', 
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Regression Diagnostic Plots', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("Diagnostic Summary:")
print(f"Mean of residuals: £{residuals.mean():.2e} (should be ≈ 0)")
print(f"SD of residuals:   £{residuals.std():.2f}")
print(f"\nThese plots all show good behaviour: no obvious patterns,")
print(f"residuals are approximately normal, and variance is roughly constant.")

## 5.6 The Gauss-Markov Assumptions

The OLS estimator is **BLUE** (Best Linear Unbiased Estimator) under five assumptions:

1. **Linearity**: The true relationship is $Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i$
   - The regression function is linear in parameters

2. **Exogeneity**: $E[\varepsilon_i | X_i] = 0$
   - No correlation between errors and regressors
   - Implies: errors have mean zero and are uncorrelated with X

3. **Homoscedasticity**: $\text{Var}(\varepsilon_i | X_i) = \sigma^2$ for all i
   - Constant error variance (does not depend on X)

4. **No Autocorrelation**: $\text{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$
   - Errors are independent (especially important in time series)

5. **No Perfect Multicollinearity**
   - In simple regression: X is not constant and has variation
   - In multiple regression: regressors are not perfectly linearly dependent

**Practical importance:**
- If all five hold → OLS is BLUE, estimates are unbiased, inference is valid
- If assumptions fail → estimates may be biased, standard errors incorrect, inference unreliable
- Always diagnose violations using residual plots and specification tests

In [ ]:
# Demonstrate violations of assumptions

# Generate three datasets:
# 1. Well-behaved data (our current df)
# 2. Heteroscedastic data (variance increases with X)
# 3. Non-linear data (quadratic relationship)

np.random.seed(42)
n = 50
X_sim = np.random.uniform(2, 50, n)

# Well-behaved
Y_good = 350000 - 1200 * X_sim + np.random.normal(0, 25000, n)

# Heteroscedastic (variance increases with X)
Y_hetero = 350000 - 1200 * X_sim + np.random.normal(0, 10000 + 1000 * X_sim, n)

# Non-linear (quadratic relationship)
Y_nonlinear = 400000 - 5000 * X_sim + 50 * X_sim**2 + np.random.normal(0, 25000, n)

# Fit regressions
def fit_and_residuals(X, Y):
    X_const = sm.add_constant(X)
    model = sm.OLS(Y, X_const)
    results = model.fit()
    return results, results.fittedvalues, results.resid

results_good, yhat_good, resid_good = fit_and_residuals(X_sim, Y_good)
results_hetero, yhat_hetero, resid_hetero = fit_and_residuals(X_sim, Y_hetero)
results_nonlinear, yhat_nonlinear, resid_nonlinear = fit_and_residuals(X_sim, Y_nonlinear)

# Plot side-by-side comparisons
fig, axes = plt.subplots(3, 2, figsize=(13, 12))

# Row 1: Well-behaved
ax = axes[0, 0]
ax.scatter(X_sim, Y_good/1000, s=60, alpha=0.6, color='green', edgecolors='darkgreen')
ax.plot(X_sim, yhat_good/1000, 'r-', linewidth=2)
ax.set_ylabel('Property Value (£000s)', fontsize=10)
ax.set_title('Well-Behaved Data', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.scatter(yhat_good/1000, resid_good/1000, s=60, alpha=0.6, color='green', edgecolors='darkgreen')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax.set_ylabel('Residuals (£000s)', fontsize=10)
ax.set_title('Residuals vs Fitted (Good)', fontsize=11, fontweight='bold')
ax.text(0.05, 0.95, '✓ Homoscedastic\n✓ Linearity', transform=ax.transAxes,
        fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
ax.grid(True, alpha=0.3)

# Row 2: Heteroscedastic
ax = axes[1, 0]
ax.scatter(X_sim, Y_hetero/1000, s=60, alpha=0.6, color='orange', edgecolors='darkorange')
ax.plot(X_sim, yhat_hetero/1000, 'r-', linewidth=2)
ax.set_ylabel('Property Value (£000s)', fontsize=10)
ax.set_title('Heteroscedastic Data (Variance ↑ with X)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.scatter(yhat_hetero/1000, resid_hetero/1000, s=60, alpha=0.6, color='orange', edgecolors='darkorange')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax.set_ylabel('Residuals (£000s)', fontsize=10)
ax.set_title('Residuals vs Fitted (Heteroscedastic)', fontsize=11, fontweight='bold')
ax.text(0.05, 0.95, '✗ Heteroscedastic\n  (Funnel pattern)', transform=ax.transAxes,
        fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='#ffe6cc', alpha=0.7))
ax.grid(True, alpha=0.3)

# Row 3: Non-linear
ax = axes[2, 0]
ax.scatter(X_sim, Y_nonlinear/1000, s=60, alpha=0.6, color='red', edgecolors='darkred')
ax.plot(X_sim, yhat_nonlinear/1000, 'r-', linewidth=2)
ax.set_xlabel('Distance (km)', fontsize=10)
ax.set_ylabel('Property Value (£000s)', fontsize=10)
ax.set_title('Non-Linear Data (Quadratic)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[2, 1]
ax.scatter(yhat_nonlinear/1000, resid_nonlinear/1000, s=60, alpha=0.6, color='red', edgecolors='darkred')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Fitted Values (£000s)', fontsize=10)
ax.set_ylabel('Residuals (£000s)', fontsize=10)
ax.set_title('Residuals vs Fitted (Non-Linear)', fontsize=11, fontweight='bold')
ax.text(0.05, 0.95, '✗ Non-linear pattern\n  (U-shaped)', transform=ax.transAxes,
        fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='#ffcccc', alpha=0.7))
ax.grid(True, alpha=0.3)

plt.suptitle('Gauss-Markov Assumption Violations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Summary of violations:")
print(f"\nHeteroscedasticity: Notice the 'funnel' pattern—residuals spread wider at high fitted values.")
print(f"This violates constant variance and inflates standard errors.")
print(f"\nNon-linearity: The U-shaped residual pattern shows the linear model misses curvature.")
print(f"The true relationship is quadratic, not linear.")

## 5.7 Connection to ANOVA

Simple regression with a binary regressor is equivalent to a two-sample t-test or one-way ANOVA.

**Example:** Instead of continuous distance, divide municipalities into "Near hub" (1) and "Far from hub" (0).

The regression coefficient β₁ then represents the **difference in means** between the two groups.

The **F-test** from regression equals the **t-test squared**:

$$F = t^2$$

This reveals the deep connection between regression and hypothesis testing about group differences.

In [ ]:
# Create binary variable: near hub (< 25 km) vs far from hub (>= 25 km)
df['near_hub'] = (df['distance_km'] < 25).astype(int)

print("Sample split:")
print(df.groupby('near_hub')['property_value'].agg(['count', 'mean', 'std']))
print(f"\nMean property value (near hub):  £{df[df['near_hub']==1]['property_value'].mean():.0f}")
print(f"Mean property value (far hub):   £{df[df['near_hub']==0]['property_value'].mean():.0f}")
print(f"Difference:                      £{df[df['near_hub']==1]['property_value'].mean() - df[df['near_hub']==0]['property_value'].mean():.0f}")

In [ ]:
# Regression with dummy variable
X_dummy = sm.add_constant(df['near_hub'].values)
model_dummy = sm.OLS(df['property_value'].values, X_dummy)
results_dummy = model_dummy.fit()

print(results_dummy.summary())

print(f"\n--- Interpretation ---")
print(f"β̂₀ (intercept): £{results_dummy.params[0]:.0f}")
print(f"  = Mean property value when near_hub = 0 (far from hub)")
print(f"\nβ̂₁ (slope): £{results_dummy.params[1]:.0f}")
print(f"  = Difference in mean property value (near minus far)")
print(f"  = £{df[df['near_hub']==1]['property_value'].mean():.0f} - £{df[df['near_hub']==0]['property_value'].mean():.0f}")

In [ ]:
# Compare regression F-test with two-sample t-test
near_values = df[df['near_hub']==1]['property_value'].values
far_values = df[df['near_hub']==0]['property_value'].values

# Two-sample t-test
t_stat_ttest, p_value_ttest = stats.ttest_ind(near_values, far_values)

# From regression
F_stat_reg = results_dummy.fvalue
p_value_reg = results_dummy.f_pvalue

print(f"Two-sample t-test:")
print(f"  t-statistic: {t_stat_ttest:.4f}")
print(f"  p-value: {p_value_ttest:.6f}")
print(f"\nRegression F-test:")
print(f"  F-statistic: {F_stat_reg:.4f}")
print(f"  p-value: {p_value_reg:.6f}")
print(f"\nVerification:")
print(f"  F = t²: {F_stat_reg:.4f} vs {t_stat_ttest**2:.4f}")
print(f"  Match: {np.isclose(F_stat_reg, t_stat_ttest**2)}")

## Exercises

### Exercise 1: Advertising Spend and Firm Revenue
A firm collects data on advertising spend (£000s) and quarterly revenue (£000s) across 20 quarters. Estimate a simple linear regression and interpret the slope coefficient.

### Exercise 2: Interpreting Low R²
Suppose you regress house prices on square footage and obtain R² = 0.15. Does this mean your model is useless? Discuss what a low R² tells us and does not tell us about model validity.

### Exercise 3: Confidence Intervals and Hypothesis Tests
From a regression output, (a) construct a 95% CI for the slope, (b) test H₀: β₁ = 0 at the 5% level, and (c) discuss whether the results support a meaningful economic effect.

### Exercise 4: Residual Diagnostics
Examine residual plots from a regression. Identify any potential violations of Gauss-Markov assumptions and suggest remedial steps.

In [ ]:
# ============================================================================
# EXERCISE STARTER CODE
# ============================================================================

# Exercise 1: Advertising Spend and Revenue
print("\n" + "="*70)
print("EXERCISE 1: Advertising Spend and Revenue")
print("="*70)

np.random.seed(42)
n_quarters = 20
ad_spend = np.random.uniform(5, 50, n_quarters)  # £000s
revenue = 150 + 8 * ad_spend + np.random.normal(0, 20, n_quarters)  # £000s
df_ad = pd.DataFrame({'ad_spend': ad_spend, 'revenue': revenue})

print("\nData (first 5 rows):")
print(df_ad.head())

# TODO: Fit OLS regression of revenue on ad_spend
# model_ad = sm.OLS(...)
# results_ad = model_ad.fit()
# print(results_ad.summary())
# Interpret: What is the effect of a £1,000 increase in advertising spend?

print("\n" + "-"*70)
print("Exercise 1: Your analysis goes here...")
print("-"*70)

In [ ]:
# Exercise 2: Interpreting Low R²
print("\n" + "="*70)
print("EXERCISE 2: Interpreting Low R²")
print("="*70)

# Scenario: You regress house prices on floor area and get R² = 0.15
print("\nScenario: A simple regression of house price on floor area yields R² = 0.15.")
print("\nQuestions:")
print("  (a) Does this mean the model is useless?")
print("  (b) What might cause a low R² even if floor area is genuinely important?")
print("  (c) Can a regression with low R² still provide valid inference on β₁?")
print("\nTodo: Write a brief paragraph addressing each question.")
print("\n" + "-"*70)
print("Exercise 2: Your analysis goes here...")
print("-"*70)

In [ ]:
# Exercise 3: Confidence Intervals and Hypothesis Tests
print("\n" + "="*70)
print("EXERCISE 3: Confidence Intervals and Hypothesis Tests")
print("="*70)

# Using the advertising data from Exercise 1
print("\nUsing the advertising spend regression:")
print("\nTasks:")
print("  (a) Construct a 95% confidence interval for the slope.")
print("  (b) Test H₀: β₁ = 0 at the 5% significance level.")
print("  (c) What does rejection of H₀ mean in economic terms?")
print("  (d) Is the estimated effect economically significant?")
print("\n" + "-"*70)
print("Exercise 3: Your analysis goes here...")
print("-"*70)

In [ ]:
# Exercise 4: Residual Diagnostics
print("\n" + "="*70)
print("EXERCISE 4: Residual Diagnostics")
print("="*70)

# Generate data with heteroscedasticity for diagnostic practice
np.random.seed(123)
X_ex4 = np.random.uniform(0, 100, 40)
Y_ex4 = 100 + 2*X_ex4 + np.random.normal(0, 5 + 0.5*X_ex4, 40)
df_ex4 = pd.DataFrame({'X': X_ex4, 'Y': Y_ex4})

# Fit the regression
X_ex4_const = sm.add_constant(X_ex4)
model_ex4 = sm.OLS(Y_ex4, X_ex4_const)
results_ex4 = model_ex4.fit()

print("\nRegression Summary:")
print(results_ex4.summary())

# TODO: Create diagnostic plots (like Section 5.5)
# - Residuals vs Fitted
# - Q-Q plot
# - Residuals vs X
# - Histogram of residuals

print("\nTasks:")
print("  (a) Create 2x2 diagnostic plots (see Section 5.5 for code).")
print("  (b) Identify which assumption(s) appear to be violated.")
print("  (c) What are the practical consequences of the violation(s)?")
print("  (d) Suggest a remedy (e.g., transformation, robust SE, different model).")
print("\n" + "-"*70)
print("Exercise 4: Your analysis goes here...")
print("-"*70)